# 🍳 Fridge Chef: Solutions Notebook

Welcome! In this notebook you'll build an **AI-powered kitchen agent** that reads your fridge, understands your cravings, and plans dinner for you.

---

## 🧠 What You'll Learn

- **Rule-based agents** — how far simple `if/else` pipelines can take you (and where they break)
- **LLM-powered agents** — using a language model to interpret natural language intent
- **Multi-agent architecture** — splitting reasoning (Thinker) and validation (Validator) into separate roles
<!-- - **Prompt engineering** — crafting system prompts that produce structured, reliable outputs
- **ML distillation** — training a small classifier to mimic an LLM's decisions -->

---

## 📋 Outline

| # | Section | What happens |
|---|---------|-------------|
| 1 | **Setup & Explore** | Install deps, load the kitchen, see your fridge |
| 2 | **Play It Yourself** | Try the interactive game manually |
| 3 | **Phase 1 — Rule-Based Agent** | Build a keyword-matching pipeline (fast but brittle) |
| 4 | **Phase 2 — LLM-Powered Agent** | Replace keyword matching with an LLM that *gets* intent |
| 5 | **Phase 3 — ML Classifier** | Distill the LLM into a tiny trained model |

---

**Phases at a glance:**
- 🟢 **Phase 1**: Rule-based think function (working)
- 🔵 **Phase 2**: LLM-powered think function (working)
- 🟡 **Phase 3**: ML classifier think function (skeleton — your turn!)

In [ ]:
#@title Fetch necessary utilities { display-mode: "form" }
import sys, os
import os
import openai
import json, re, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/fridge_chef')

sys.path.insert(0, '.')

!pip install openai --quiet
print('Setup complete!')

Setup complete!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#@title Load the game and preview the kitchen { display-mode: "form" }
from fridge_chef import *
from fridge_chef.game import KitchenWorld
from fridge_chef.data import ChefState
from fridge_chef.agent import TOOLS_DESCRIPTION, parse_action
print('Fridge Chef loaded!')

# Preview the kitchen
chef, world, tools = create_game()
print()
print(world.fridge_summary())
print()
print('Available tools:')
print(TOOLS_DESCRIPTION)

Fridge Chef loaded!

Fridge (12 items): green onions, sugar, lemon juice, curry powder, paprika, black pepper, soy sauce, mayonnaise, basil, parmesan cheese, egg, butter

Available tools:
Available tools (use exactly one per turn):

ACTION: check_fridge()
  See what ingredients are currently in your fridge.

ACTION: search_recipes(ingredient="eggs")
  Search for recipes that use a specific ingredient from your fridge.

ACTION: get_ingredients(recipe="omelette")
  Get the full ingredient list for a recipe. This also sets it as your
  chosen recipe. Compare against your fridge to see what's missing.

ACTION: add_to_shopping_list(item="paprika")
  Add a missing ingredient to your shopping list. Only add items you
  don't already have in the fridge.

ACTION: done()
  Signal that you're finished — recipe chosen and shopping list ready.
  Make sure you've chosen a recipe with get_ingredients() first!


---
## 🎮 Play the Game Yourself!

In [ ]:
#@title Play the game interactively { display-mode: "form" }
from fridge_chef.interactive import play_interactive

game = play_interactive()

---
## 🟢 Phase 1 — Rule-Based Think Function

Strategy: follow a linear pipeline to fulfill the user's request:

1. **Check fridge** — see what we have
2. **Extract ingredient** — scan the request for a fridge item by keyword
3. **Search recipes** — find recipes using that ingredient
4. **Pick recipe** — choose the first match
5. **Get ingredients** — see what the recipe needs
6. **Add missing items** — add each one to the shopping list
7. **Done** — signal completion

**The fundamental limitation**: step 2 only works when the user names an ingredient directly
(`"I want to cook eggs"`). For natural requests like `"something comforting for a cold evening"`,
no keyword matches — so the agent blindly falls back to the first fridge item.
The pipeline still *completes*, but the recipe choice is meaningless.

In [5]:
def _extract_ingredient(user_request: str, fridge: list[str]) -> str:
    """Try to find a fridge item mentioned in the request.

    Works for: "I want to cook something with eggs" → "eggs"
    Fails for: "something comforting for a cold evening" → falls back to fridge[0]

    This is the hard ceiling of the rule-based approach — interpreting intent
    requires language understanding, not string matching.
    """
    request_lower = user_request.lower()
    for item in fridge:
        if item.lower() in request_lower:
            return item
    # No match — fall back to first fridge item regardless of what the user wants
    return fridge[0] if fridge else "eggs"


def think_rule_based(chef: ChefState, world: KitchenWorld, history: list[dict], user_request: str) -> str:
    """Rule-based kitchen agent following a linear pipeline."""

    # --- Step 1: Check fridge if we haven't yet ---
    if not chef.fridge_contents:
        return 'ACTION: check_fridge()'

    # --- Step 2: Search recipes if we haven't yet ---
    if not chef.possible_recipes:
        ingredient = _extract_ingredient(user_request, chef.fridge_contents)
        return f'ACTION: search_recipes(ingredient="{ingredient}")'

    # --- Step 3: Pick a recipe and get its ingredients ---
    if not chef.chosen_recipe:
        recipe = chef.possible_recipes[0]
        return f'ACTION: get_ingredients(recipe="{recipe}")'

    # --- Step 4: Add missing ingredients to shopping list one by one ---
    if chef.needed_ingredients:
        fridge_lower = [item.lower() for item in chef.fridge_contents]
        shopping_lower = [item.lower() for item in chef.shopping_list]
        for ingredient in chef.needed_ingredients:
            if ingredient.lower() not in fridge_lower and ingredient.lower() not in shopping_lower:
                return f'ACTION: add_to_shopping_list(item="{ingredient}")'

    # --- Step 5: All done! ---
    return 'ACTION: done()'

In [ ]:
USER_REQUEST = "I want something comforting for a cold evening" # @param {type:"string"}

In [ ]:
#@title Run the rule-based agent { display-mode: "form" }
result_rb = play_rule_based(think_rule_based, user_request=USER_REQUEST, max_turns=20, delay=0.05)
print(f"\nRequest:  '{USER_REQUEST}'")
print(f"Result:   {'Complete!' if result_rb['completed'] else 'Timed out'}")
print(f"Recipe:   {result_rb['recipe']}  ← picked because it was first in the list, not because it's comforting")
print(f"Shopping: {', '.join(result_rb['shopping_list']) or 'nothing needed'}")
print(f"Turns:    {result_rb['turns']}")


Request:  'I want something comforting for a cold evening'
Result:   Complete!
Recipe:   omelette  ← picked because it was first in the list, not because it's comforting
Shopping: salt, pepper
Turns:    7


---
## 🔵 Phase 2 — LLM-Powered Think Function

The rule-based agent broke on `"I want something comforting for a cold evening"` — it ignored
the intent and grabbed the first fridge item. The LLM solves this at the only point where it matters:
**choosing which ingredient to search for**.

Everything else (checking the fridge, adding items, calling done) is still autopilot.
The LLM's one job is to read the natural language request and decide: *"comforting + cold evening
→ chicken curry or a warm stir fry → search for chicken"*.

That single reasoning step is what no amount of `if/else` can replicate.

In [ ]:
os.environ['OPENAI_API_KEY'] = 'your-key-here'
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')

client = openai.OpenAI(
    api_key=api_key,
    base_url="https://litellm.sph-prod.ethz.ch/v1",
)
MODEL = "anthropic/claude-sonnet-4-5"
print('LLM client ready!')

LLM client ready!


## 🔄 Kitchen Agent Workflow
```
                    User request
                         │
                         ▼
              ┌─────────────────────────┐
              │      🧠 Thinker        │
              │  Interprets intent,     │
              │   proposes action       │
              └─────────────────────────┘
                         │ proposed action
                         ▼
              ┌─────────────────────────┐
              │      🔍 Validator      │◄─────────┐
              │  Cross-checks fridge    │  revise  │
              │    + shopping list      │──────────┘
              └─────────────────────────┘
                         │ final action
                         ▼
              ┌─────────────────────────┐
              │      🌍 World          │
              │  Executes action,       │
              │    updates state        │
              └─────────────────────────┘
                         │
               ┌─────────┴──────────┐
               ▼                    ▼
           done()              continue
               │                    │
               ▼                    └──► (next turn → Thinker)
    recipe + shopping list
           complete
```

**Each turn:**
1. 🧠 **Thinker** — reads the full kitchen state, interprets the user's natural language intent, proposes the next action
2. 🔍 **Validator** — cross-references the fridge contents and shopping list, approves or revises the action
3. 🌍 **World** — executes the final validated action and updates the state
4. Repeat until `done()`

#### 🎯 TODO 1 — Write the Thinker’s System Prompt

##### Objective  
Design the **Thinker** so it understands user intent and executes a structured workflow.

---

##### Requirements

The **Thinker must:**

- **Understand the user’s intent** — not just match keywords  
  - Example:  
    - *“something warming for winter”* → search for `"chicken"` (not random ingredients)

- **Follow this exact 5-step workflow:**
  1. `check_fridge()`
  2. `search_recipes(ingredient="...")`
  3. `get_ingredients(recipe="...")`
  4. `add_to_shopping_list(item="...")`  
     →  Add **one item at a time**
  5. `done()`

- **Output exactly two lines:**
  ```text
  REASONING: <one sentence>
  ACTION: <tool_call>

In [ ]:
 # ─────────────────────────────────────────────────────────────────────────────
# Prompts
# ─────────────────────────────────────────────────────────────────────────────
 
THINKER_PROMPT = """You are the Thinker — a creative kitchen planning agent.
 
YOUR JOB: Interpret the user's natural language request and decide the single best next action.
 
WORKFLOW (follow in order):
  1. If fridge contents are unknown → check_fridge()
  2. If no recipes found yet → search_recipes(ingredient="<ingredient that best captures the user's intent>")
     • Pick the ingredient that BEST matches the user's mood/intent — not just the first fridge item.
     • "warming winter dinner"  → chicken or rice
     • "light summer lunch"     → tomatoes or cucumber
     • "quick weeknight meal"   → eggs or pasta
  3. If recipes found but none chosen → get_ingredients(recipe="<recipe that best fits the request>")
  4. If recipe chosen → add_to_shopping_list(item="<one item NOT in fridge and NOT already on shopping list>")
  5. If all missing ingredients are covered → done()
 
AVAILABLE TOOLS:
{tools}
 
OUTPUT FORMAT — always exactly two lines, nothing else:
REASONING: <one sentence explaining your choice and why it fits the user's intent>
ACTION: <tool_call>
"""
 
VALIDATOR_PROMPT = """You are the Validator — a precise kitchen safety agent with full situational awareness.
 
YOUR JOB: Given the complete kitchen state, verify the Thinker's proposed action before it reaches the world.
 
CHECK ALL OF THESE IN ORDER:
  1. FRIDGE DUPLICATES — Is the proposed item already in the fridge (exact or near-match)?
       "garlic" proposed + fridge has "garlic"        → REVISE to the next truly missing item
       "double cream" proposed + fridge has "cream"   → likely duplicate, REVISE
       "spring onion" proposed + fridge has "onion"   → different ingredient, APPROVE
  2. SHOPPING LIST DUPLICATES — Is the item already on the shopping list?
  3. WORKFLOW ORDER — Is this the right step given the current state?
       Don't search recipes before fridge is known.
       Don't add ingredients before a recipe is chosen.
  4. INTENT ALIGNMENT — Does this action actually serve the user's request?
       Don't pick a meat recipe if the user asked for vegetarian.
       Don't pick a heavy dish if the user asked for something light.
  5. DONE SAFETY — If the action is done(), verify EVERY recipe ingredient is either
       in the fridge OR on the shopping list. If anything is missing, REVISE to add it instead.
 
Respond with ONLY valid JSON (no markdown, no extra text):
{"verdict": "APPROVE" or "REVISE", "action": "<final action string>", "reason": "<one sentence>"}
 
If APPROVE → copy the Thinker's action unchanged into "action".
If REVISE  → put your corrected action into "action".
"""

In [ ]:
#@title Define helpers for parsing LLM responses and calling the API { display-mode: "form" }

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────
 
def parse_reasoning(text: str) -> str:
    m = re.search(r"REASONING:\s*(.+)", text, re.I)
    return m.group(1).strip() if m else "—"
 
 
def parse_validator_json(text: str) -> dict:
    try:
        return json.loads(re.sub(r"```json|```", "", text).strip())
    except Exception:
        verdict  = "APPROVE" if "APPROVE" in text else "REVISE"
        action_m = re.search(r'"action"\s*:\s*"([^"]+)"', text)
        reason_m = re.search(r'"reason"\s*:\s*"([^"]+)"', text)
        return {
            "verdict": verdict,
            "action":  action_m.group(1) if action_m else None,
            "reason":  reason_m.group(1) if reason_m else "could not parse validator response",
        }
 
 
def call_llm(system: str, user: str, temperature: float = 0.3) -> str:
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            max_tokens=400,
            temperature=temperature,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"  [API ERROR] {e}")
        return ""
 


#### 🎯 TODO 2 — Format `parse_action` Output into ACTION String

##### Objective  
Convert the output of `parse_action(text)` into a properly formatted `ACTION:` string.

---

##### Input Format

`parse_action(text)` returns a **tuple**:

```python
(tool_name, args_dict)

In [ ]:
# ── 🎯 TODO 2 ───────────────────────────────────────────────────────────────────
# Convert parse_action's output into a proper ACTION: string.

# Hint: use ", ".join(f'{k}="{v}"' for k, v in args.items()) for the args
#       and return "ACTION: check_fridge()" as fallback if tool_name is empty
# ─────────────────────────────────────────────────────────────────────────────

def build_action_string(thinker_raw: str) -> str:
    """Convert parse_action tuple output into an ACTION: string."""
    tool_name, args = parse_action(thinker_raw)
    if tool_name:
        args_str = ", ".join(f'{k}="{v}"' for k, v in args.items())
        return f"ACTION: {tool_name}({args_str})"
    return "ACTION: check_fridge()"
 

In [ ]:
def build_context(chef: ChefState, history: list[dict], user_request: str) -> str:
    """Structured snapshot of the full kitchen state shared by both agents."""
    lines = [f'USER REQUEST: "{user_request}"', ""]
 
    lines.append(f"FRIDGE CONTENTS: {', '.join(chef.fridge_contents) if chef.fridge_contents else '(not checked yet)'}")
    lines.append(f"POSSIBLE RECIPES: {', '.join(chef.possible_recipes) if chef.possible_recipes else '(none yet)'}")
    lines.append(f"CHOSEN RECIPE: {chef.chosen_recipe or '(none yet)'}")
 
    if chef.chosen_recipe and chef.needed_ingredients:
        fridge_lower   = [i.lower() for i in chef.fridge_contents]
        shopping_lower = [i.lower() for i in chef.shopping_list]
        have       = [i for i in chef.needed_ingredients if i.lower() in fridge_lower]
        on_list    = [i for i in chef.needed_ingredients if i.lower() in shopping_lower]
        still_need = [i for i in chef.needed_ingredients
                      if i.lower() not in fridge_lower and i.lower() not in shopping_lower]
        lines.append("INGREDIENT STATUS:")
        lines.append(f"  Already have  : {', '.join(have) or '(none)'}")
        lines.append(f"  On list       : {', '.join(on_list) or '(empty)'}")
        lines.append(f"  Still need    : {', '.join(still_need) or '(all covered!)'}")
 
    lines.append(f"SHOPPING LIST: {', '.join(chef.shopping_list) if chef.shopping_list else '(empty)'}")
 
    recent = [h for h in history[-8:] if h["role"] in ("action", "result")]
    if recent:
        lines.append("\nRECENT HISTORY:")
        for h in recent[-6:]:
            prefix = "  Action" if h["role"] == "action" else "  Result"
            lines.append(f"{prefix}: {h['content'][:120]}")
 
    return "\n".join(lines)
 

def world_execute(action: str, chef: ChefState, world: KitchenWorld) -> str:
    """Execute an action string against the world manually."""
    # Strip ACTION: prefix if present, and any leading/trailing whitespace
    action_str = re.sub(r"^ACTION:\s*", "", action.strip(), flags=re.I).strip()

    if action_str == "check_fridge()":
        chef.fridge_contents = world.get_fridge_contents()
        return f"Fridge contains: {', '.join(chef.fridge_contents)}"

    m = re.search(r'search_recipes\(ingredient="?([^")]+)"?\)', action_str)
    if m:
        ingredient = m.group(1)
        chef.possible_recipes = world.search_recipes(ingredient)
        return f"Found recipes: {', '.join(chef.possible_recipes)}"

    m = re.search(r'get_ingredients\(recipe="?([^")]+)"?\)', action_str)
    if m:
        recipe = m.group(1)
        chef.chosen_recipe = recipe
        chef.needed_ingredients = world.get_recipe_ingredients(recipe)
        return f"Ingredients for '{recipe}': {', '.join(chef.needed_ingredients)}"

    m = re.search(r'add_to_shopping_list\(item="?([^")]+)"?\)', action_str)
    if m:
        item = m.group(1)
        if item.lower() not in [s.lower() for s in chef.shopping_list]:
            chef.shopping_list.append(item)
        return f"Added '{item}' → shopping list: {', '.join(chef.shopping_list)}"

    if action_str == "done()":
        return "Workflow complete!"

    return f"Unknown action: {action_str}"
 

#### 🎯TODO 4 — Wire the two agents together

Implement `think_llm()` — the function that connects the Thinker and Validator into a single decision step.

**Steps:**

**A.** Build the shared context using `build_context()`

**B.** Call the Thinker LLM → get `proposed_action` via `build_action_string()`

**C.** Call the Validator LLM → parse its JSON with `parse_validator_json()`

**D.** Extract the final action from the validator's response

**E.** Ensure the final action always starts with `ACTION:`

**F.** Return the final action string

---

**Hints:**

- **Step C** — pass the context + `proposed_action` to the Validator, and use `temperature=0.1` so it is consistent and deterministic
- **Step D** — use `v.get("action")` and fall back to `proposed_action` if the validator's JSON is malformed

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# think_llm  
# ─────────────────────────────────────────────────────────────────────────────
 
def think_llm(
    chef: ChefState,
    world: KitchenWorld,
    history: list[dict],
    user_request: str,
) -> str:
    """
    Two-agent think function.
    1. Thinker  — interprets user intent, proposes an action
    2. Validator — cross-references fridge + shopping list, approves or corrects
    Returns the final validated action string.
    """
    context         = build_context(chef, history, user_request)
    thinker_raw     = call_llm(system=THINKER_PROMPT.format(tools=TOOLS_DESCRIPTION), user=context)
    proposed_action = build_action_string(thinker_raw)
 
    validator_raw = call_llm(
        system=VALIDATOR_PROMPT,
        user=f"{context}\n\nThinker's proposed action: {proposed_action}\n\nCross-check against fridge and shopping list.",
        temperature=0.1,
    )
    v            = parse_validator_json(validator_raw)
    final_action = v.get("action") or proposed_action
    if not final_action.upper().startswith("ACTION:"):
        final_action = f"ACTION: {final_action}"
 
    return final_action
 

In [ ]:

 
# ─────────────────────────────────────────────────────────────────────────────
# run_with_thoughts  (standalone — prints full reasoning each turn)
# ─────────────────────────────────────────────────────────────────────────────
 
def run_with_thoughts(user_request: str, max_turns: int = 5):
    chef, world, tools = create_game()
    history = []
    final_action = ""
 
    print(f"\n{'═'*55}")
    print(f"  🍽  Request: {user_request}")
    print(f"{'═'*55}")
 
    for turn in range(max_turns):
        print(f"\n┌─ Turn {turn + 1} {'─'*45}")
 
        # ── Thinker ──
        context         = build_context(chef, history, user_request)
        thinker_raw     = call_llm(system=THINKER_PROMPT.format(tools=TOOLS_DESCRIPTION), user=context)
        thinker_reason  = parse_reasoning(thinker_raw)
        proposed_action = build_action_string(thinker_raw)
 
        print(f"│ 🧠 THINKER   {thinker_reason}")
        print(f"│              → {proposed_action.replace('ACTION: ', '')}")
 
        # ── Validator ──
        validator_raw = call_llm(
            system=VALIDATOR_PROMPT,
            user=f"{context}\n\nThinker's proposed action: {proposed_action}\n\nCross-check against fridge and shopping list.",
            temperature=0.1,
        )
        v            = parse_validator_json(validator_raw)
        final_action = v.get("action") or proposed_action
        reason       = v.get("reason", "—")
        was_revised  = final_action.strip() != proposed_action.strip()
 
        if not final_action.upper().startswith("ACTION:"):
            final_action = f"ACTION: {final_action}"
 
        if was_revised:
            print(f"│ 🔍 VALIDATOR ↻ REVISED  — {reason}")
            print(f"│              → {final_action.replace('ACTION: ', '')}")
        else:
            print(f"│ 🔍 VALIDATOR ✓ APPROVED — {reason}")
 
        # ── Execute ──
        result = world_execute(final_action, chef, world)
        print(f"│ 🌍 WORLD     {result}")
        print(f"└{'─'*51}")
 
        history.append({"role": "action", "content": final_action})
        history.append({"role": "result", "content": result})
 
        if "done()" in final_action:
            break
 
    print(f"\n{'═'*55}")
    print(f"  {'✅ Complete!' if 'done()' in final_action else '⏱ Timed out'}")
    print(f"  Recipe:   {chef.chosen_recipe or '—'}")
    print(f"  Shopping: {', '.join(chef.shopping_list) or 'nothing needed'}")
    print(f"  Turns:    {turn + 1}")
    print(f"{'═'*55}\n")
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────────────────────────────────────
 
USER_REQUEST = "something comforting for a cold winter evening"
run_with_thoughts(USER_REQUEST)


═══════════════════════════════════════════════════════
  🍽  Request: something comforting for a cold winter evening
═══════════════════════════════════════════════════════

┌─ Turn 1 ─────────────────────────────────────────────
│ 🧠 THINKER   I need to first check what ingredients are available in the fridge before I can plan a comforting winter meal.
│              → check_fridge()
│ 🔍 VALIDATOR ↻ REVISED  — Correct first step: must know fridge contents before searching recipes or adding ingredients.
│              → check_fridge()
│ 🌍 WORLD     Fridge contains: sugar, tomatoes, vanilla extract, butter, coconut milk, pasta, cumin, garlic, parmesan cheese, curry powder, tortilla, mustard
└───────────────────────────────────────────────────

┌─ Turn 2 ─────────────────────────────────────────────
│ 🧠 THINKER   For a comforting cold winter evening, pasta is the most warming and satisfying ingredient in the fridge that fits the cozy comfort food intent.
│              → search_recipes(i

In [ ]:
# Run this first to see the available methods
chef, world, tools = create_game()
print([m for m in dir(world) if not m.startswith('_')])

['fridge', 'fridge_summary', 'get_fridge_contents', 'get_recipe_ingredients', 'ingredients_db', 'recipe_db', 'search_recipes']


In [ ]:
test_response = call_llm(
    system="You are a helpful assistant.",
    user="Say hello and nothing else.",
)
print(f"Test response: '{test_response}'")

  [RAW RESPONSE] ChatCompletion(id='chatcmpl-a483f08e-49e4-43e0-a5bf-c2d83526b70c', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1773716649, model='anthropic/claude-sonnet-4-5', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=5, prompt_tokens=19, total_tokens=24, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=5), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0, cache_creation_tokens=0, cache_creation_token_details={'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}), cache_creation_input_tokens=0, cache_read_input_tokens=0))
  [CONTENT] Hello!
Test response: 'Hello!'


In [ ]:

result_llm = play_with_llm(
    think_fn=lambda chef, world, history, req: think_llm(chef, world, history, req),
    user_request=USER_REQUEST,
    max_turns=20,
    delay=0.5,
)

print(f"\nRequest: '{USER_REQUEST}'")
print(f"Result:  {'Complete!' if result_llm['completed'] else 'Timed out'}")
print(f"Recipe:  {result_llm['recipe']}  ← chosen because it matches the intent")
print(f"Shopping: {', '.join(result_llm['shopping_list']) or 'nothing needed'}")
print(f"Turns:   {result_llm['turns']}")

  [RAW RESPONSE] ChatCompletion(id='chatcmpl-d830d326-5564-4740-9df8-3b3432deb3bd', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='REASONING: For a comforting cold winter evening meal, chicken breast is the most substantial and warming ingredient in the fridge, perfect for hearty winter dishes like soups, stews, or baked chicken.\nACTION: search_recipes(ingredient="chicken breast")', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1773716206, model='anthropic/claude-sonnet-4-5', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=60, prompt_tokens=668, total_tokens=728, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=60), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0,

KeyboardInterrupt: 

In [ ]:
result_llm = play_with_llm(
    think_fn=lambda chef, world, history, req: think_llm(chef, world, history, req),
    user_request=USER_REQUEST,
    max_turns=20,
    delay=0.5,
)
print(f"\nRequest: '{USER_REQUEST}'")
print(f"Result:  {'Complete!' if result_llm['completed'] else 'Timed out'}")
print(f"Recipe:  {result_llm['recipe']}  ← chosen because it matches the intent")
print(f"Shopping: {', '.join(result_llm['shopping_list']) or 'nothing needed'}")
print(f"Turns:   {result_llm['turns']}")


Request: 'I want something comforting for a cold evening'
Result:  Complete!
Recipe:  chicken curry  ← chosen because it matches the intent
Shopping: coconut milk, curry powder, ginger
Turns:   8


In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────

print(f"Request: '{USER_REQUEST}'")
print()
print(f"{'':25} {'RULE-BASED':>20}    {'LLM':>20}")
print(f"{'─'*70}")
print(f"{'Ingredient searched':25} {result_rb.get('_ingredient', 'eggs (fallback)'):>20}    {'(interpreted from request)':>20}")
print(f"{'Recipe chosen':25} {result_rb['recipe']:>20}    {result_llm['recipe']:>20}")
print(f"{'Shopping list size':25} {len(result_rb['shopping_list']):>20}    {len(result_llm['shopping_list']):>20}")
print(f"{'Turns used':25} {result_rb['turns']:>20}    {result_llm['turns']:>20}")
print()
print("Rule-based: ignored the request, picked the first recipe alphabetically.")
print("LLM:        interpreted the intent, picked a recipe that actually fits.")

Request: 'I want something comforting for a cold evening'

                                    RULE-BASED                     LLM
──────────────────────────────────────────────────────────────────────
Ingredient searched            eggs (fallback)    (interpreted from request)
Recipe chosen                         omelette           chicken curry
Shopping list size                           2                       3
Turns used                                   7                       8

Rule-based: ignored the request, picked the first recipe alphabetically.
LLM:        interpreted the intent, picked a recipe that actually fits.


In [ ]:
# Download the game log - 
# NO THIS IS NOT THE WAY... TO GENERATE A MEANINGFUL DATASET ITS BETTER TO LET 2 AGENTS PLAY AGAINST EACH OTHER AND THEN DOWNLOAD THE LOGS
try:
    from google.colab import files
    files.download(result['log_file'])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print('Open the file to inspect your agent\'s decisions turn by turn.')

---
## 🟡 Phase 3: ML Classifier Think Function (WORK IN PROGRESS)

**Goal**: Replace the LLM with a trained classifier that predicts which tool to call.
This is the "distillation" step — we train a small model to mimic the LLM's behavior.

**Key steps**:
1. 📊 Generate synthetic training data from LLM game logs
2. 🏋️ Train a classifier to predict `tool_name` from state features
3. 🔧 Use a heuristic to extract tool arguments (e.g., ingredient names)

This is a **skeleton** — the structure is complete but the model isn't trained.
Students fill in the training loop and feature engineering.

In [ ]:
import json
import glob
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: Generate synthetic training data
# ═══════════════════════════════════════════════════════════════════════════
#
# Run the LLM agent multiple times with different free-form requests to build
# a dataset of (state, action) pairs.
#
# Note: the requests are intentionally vague — the LLM interprets them.
# This is what we want the classifier to eventually learn to replicate.

TRAINING_REQUESTS = [
    "I want something comforting for a cold evening",
    "Something quick I can throw together after work",
    "I'm trying to eat a bit lighter this week",
    "I want to impress someone for dinner tonight",
    "Something warm and filling for a rainy day",
    "I want to use up whatever needs to go first",
    "Something my kids will actually eat",
    "A proper hearty dinner, not a snack",
    "I feel like Asian food tonight",
    "Something simple, I don't want to spend long cooking",
]


def generate_training_data(requests=TRAINING_REQUESTS, max_turns=20):
    """Run the LLM agent on multiple requests and collect training data.

    TODO: Implement this function.

    Returns:
        list of dicts, each with:
            - 'features': dict of state features (see extract_features below)
            - 'label': str tool name that was chosen
            - 'args': dict of arguments passed to the tool
    """
    # training_data = []
    #
    # for request in requests:
    #     result = play_with_llm(
    #         think_fn=lambda c, w, h, r: think_llm(c, w, h, r),
    #         user_request=request,
    #         max_turns=max_turns,
    #         show_display=False,
    #     )
    #
    #     with open(result['log_file']) as f:
    #         log = json.load(f)
    #
    #     for turn_data in log['turns']:
    #         if 'action' not in turn_data:
    #             continue
    #         tool_name, args = parse_action(f"ACTION: {turn_data['action']}")
    #         features = {
    #             'has_fridge': 1 if turn_data.get('fridge_checked', False) else 0,
    #             'has_recipes': 1 if turn_data.get('possible_recipes') else 0,
    #             'has_chosen': 1 if turn_data.get('recipe') else 0,
    #             'shopping_count': len(turn_data.get('shopping_list', [])),
    #             'turn': turn_data['turn'],
    #         }
    #         training_data.append({
    #             'features': features,
    #             'label': tool_name,
    #             'args': args,
    #         })
    #
    # return training_data

    raise NotImplementedError("TODO: Run LLM agent to generate training data")


# ═══════════════════════════════════════════════════════════════════════════
# STEP 2: Feature extraction
# ═══════════════════════════════════════════════════════════════════════════
#
# Convert the chef state into a fixed-size feature vector for the classifier.
# The features should capture where we are in the pipeline.
#
# Note: the classifier only learns WHICH tool to call (check_fridge,
# search_recipes, get_ingredients, add_to_shopping_list, done).
# It cannot learn to interpret free-form requests — that's why argument
# extraction (step 4) still uses heuristics.

TOOL_NAMES = ['check_fridge', 'search_recipes', 'get_ingredients',
              'add_to_shopping_list', 'done']


def extract_features(chef: ChefState) -> np.ndarray:
    """Convert chef state into a feature vector for the classifier.

    TODO: Implement this function.

    Features to consider:
        - has_fridge_contents (0/1): Have we checked the fridge?
        - has_possible_recipes (0/1): Have we searched for recipes?
        - has_chosen_recipe (0/1): Have we picked a recipe?
        - num_needed (int): How many total ingredients does the recipe need?
        - num_have (int): How many ingredients do we already have?
        - num_missing (int): How many are still missing (not in fridge or shopping list)?
        - num_shopping (int): How many items are on the shopping list?

    Returns:
        np.ndarray of shape (7,) with the features above
    """
    # has_fridge = 1 if chef.fridge_contents else 0
    # has_recipes = 1 if chef.possible_recipes else 0
    # has_chosen = 1 if chef.chosen_recipe else 0
    # num_needed = len(chef.needed_ingredients)
    #
    # fridge_lower = [f.lower() for f in chef.fridge_contents]
    # shopping_lower = [s.lower() for s in chef.shopping_list]
    # num_have = sum(1 for i in chef.needed_ingredients if i.lower() in fridge_lower)
    # num_missing = sum(
    #     1 for i in chef.needed_ingredients
    #     if i.lower() not in fridge_lower and i.lower() not in shopping_lower
    # )
    # num_shopping = len(chef.shopping_list)
    #
    # return np.array([has_fridge, has_recipes, has_chosen,
    #                  num_needed, num_have, num_missing, num_shopping])

    raise NotImplementedError("TODO: Extract features from chef state")


# ═══════════════════════════════════════════════════════════════════════════
# STEP 3: Train the classifier
# ═══════════════════════════════════════════════════════════════════════════

def train_classifier(training_data):
    """Train a classifier on the collected training data.

    TODO: Implement this function.

    Args:
        training_data: list of dicts with 'features' and 'label' keys

    Returns:
        A trained sklearn model with a .predict() method
    """
    # from sklearn.tree import DecisionTreeClassifier
    #
    # X = np.array([list(d['features'].values()) for d in training_data])
    # y = np.array([TOOL_NAMES.index(d['label']) for d in training_data])
    #
    # model = DecisionTreeClassifier(max_depth=5)
    # model.fit(X, y)
    # print(f"Training accuracy: {model.score(X, y):.1%}")
    # return model

    raise NotImplementedError("TODO: Train a classifier")


# ═══════════════════════════════════════════════════════════════════════════
# STEP 4: Argument extraction heuristic
# ═══════════════════════════════════════════════════════════════════════════
#
# The classifier predicts WHICH tool to call — but not the arguments.
# For search_recipes we still fall back to the first fridge item
# (same limitation as the rule-based agent). That's fine: the classifier
# is only learning the pipeline structure, not intent interpretation.

def extract_args(tool_name: str, chef: ChefState, user_request: str) -> dict:
    """Determine tool arguments using heuristics.

    TODO: Implement this function.
    """
    # if tool_name == 'search_recipes':
    #     request_lower = user_request.lower()
    #     for item in chef.fridge_contents:
    #         if item.lower() in request_lower:
    #             return {'ingredient': item}
    #     return {'ingredient': chef.fridge_contents[0] if chef.fridge_contents else 'eggs'}
    #
    # if tool_name == 'get_ingredients':
    #     if chef.possible_recipes:
    #         return {'recipe': chef.possible_recipes[0]}
    #     return {'recipe': 'omelette'}
    #
    # if tool_name == 'add_to_shopping_list':
    #     fridge_lower = [f.lower() for f in chef.fridge_contents]
    #     shopping_lower = [s.lower() for s in chef.shopping_list]
    #     for ing in chef.needed_ingredients:
    #         if ing.lower() not in fridge_lower and ing.lower() not in shopping_lower:
    #             return {'item': ing}
    #     return {'item': ''}
    #
    # return {}

    raise NotImplementedError("TODO: Extract arguments for the predicted tool")


# ═══════════════════════════════════════════════════════════════════════════
# STEP 5: The think function using the trained classifier
# ═══════════════════════════════════════════════════════════════════════════

# model = None  # Set this after training: model = train_classifier(data)


def think_classifier(chef: ChefState, world: KitchenWorld, history: list[dict], user_request: str) -> str:
    """ML classifier think function — predicts tool from state features.

    TODO: Implement after completing steps 1-4.

    This function:
    1. Extracts features from the current chef state
    2. Uses the trained classifier to predict which tool to call
    3. Uses heuristics to determine the tool's arguments
    4. Returns the ACTION string
    """
    # features = extract_features(chef)
    # prediction = model.predict(features.reshape(1, -1))[0]
    # tool_name = TOOL_NAMES[prediction]
    # args = extract_args(tool_name, chef, user_request)
    # args_str = ', '.join(f'{k}="{v}"' for k, v in args.items())
    # return f'ACTION: {tool_name}({args_str})'

    raise NotImplementedError(
        "TODO: Complete steps 1-4, train the model, then implement this function"
    )

In [ ]:
# Once you've implemented all 5 steps above, uncomment and run:
#
# # Generate training data (requires GEMINI_API_KEY)
# data = generate_training_data()
# print(f"Collected {len(data)} training examples")
#
# # Train the classifier
# model = train_classifier(data)
#
# # Test it!
# result = play_rule_based(
#     think_fn=think_classifier,
#     user_request="I want to cook something with eggs",
#     max_turns=20,
#     delay=0.05,
# )
# print(f"\nResult: {'Complete!' if result['completed'] else 'Timed out'} | "
#       f"Recipe: {result['recipe']} | "
#       f"Shopping: {', '.join(result['shopping_list']) or 'nothing'} | "
#       f"Turns: {result['turns']}")